# Lighter-Weight Fact Verification Approach -> Sentence-Level 

This notebook evalutes sentence-level candidates for the fact verification. 

What this notebook does:

1. Creates the sentences and loads the annotated data
2. Perform NER pre-filter
3. Examine the exact-match results
4. Examine String-based comparison results + False positive cases
5. Examine Embedding similarity results + False positive cases
6. Examine Paraphrase detection results + False positive cases
7. Examine NLI results for both small and large models + False positive cases

All helper functions are placed inside `fact_check_utils.py` (imported in this notebook as `fc`).


In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import spacy
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer, CrossEncoder
from tqdm import tqdm
from scipy.special import softmax
import fact_check_utils as fc

## Paths

Here the path to the data are mentioned. One path leads to the JSON file, and the other to the text extracted from the award documents.

In [ ]:
json_file_path = r"...\annotated_facts_2026-03-31.json"
file_path      = r"...\text_test"

## 1. Load annotated data

In [ ]:
full_data_json = fc.read_json(json_file_path) 
info_results_json = fc.json_feature(full_data_json) # capturing the full information from JSON

all_facts   = fc.build_all_facts(info_results_json)
gold_labels = fc.build_gold_labels(info_results_json)   # shared by every method: 1=verified, 0=contradicted
print(f"{len(all_facts)} facts present")

## 2. NER preprocessing

In [ ]:
nlp_trans_trf = spacy.load("en_core_web_trf", disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])
nlp_trans_lg  = spacy.load("en_core_web_lg",  disable=["tagger", "parser", "lemmatizer", "attribute_ruler"])

In [ ]:
# build the NER cache, then save it. Then just load the pickle below.
per_file_ner = fc.ner_per_file(info_results_json, file_path, nlp_trans_trf)
with open("check_per_file_ner_trf.pkl", "wb") as ner_sents:
     pickle.dump(per_file_ner, ner_sents)

In [ ]:
with open("check_per_file_ner_trf.pkl", "rb") as check_ner_sents:
    per_file_ner = pickle.load(check_ner_sents)

In [ ]:
# Keep only sentences containing at least one named entity
per_file_ner_filtered = fc.filter_ner_sentences(per_file_ner)

## 3. Exact Match Baseline

In [ ]:
# fc.exact_match_score = substring exact match (1 if fact appears verbatim in a sentence)
exact_scores = []
gold_labels = gold_labels 

for info in info_results_json:
    facts = info["fact"]
    actual_path = os.path.join(file_path, info["text_name"])
    if actual_path not in per_file_ner:
        print("WRONG FILE")
        continue

    sentences = per_file_ner_filtered[actual_path]
    score = fc.exact_match_score(facts, sentences)
    exact_scores.append(score)

    if score == 1:
        print(f"MATCH FOUND in {actual_path}")
        print(f"  Fact: {facts}")
        # Find which sentence matched
        for sent in sentences:
            if facts.strip().lower() in sent.strip().lower():
                print(f"  Sentence: {sent}")
                break
    print()

print(f"\nTotal matches: {sum(exact_scores)} out of {len(exact_scores)}")
print(classification_report(gold_labels, exact_scores, digits=6, target_names=["Contradicted", "Verified"]))

## 4. Lexical similarity — Levenshtein

In [ ]:
lev_chosen_thresh = [0.0, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 1]
inspect_thresh_lev = 0.5
fp_inspect_lev = []

for thresh in lev_chosen_thresh:
    predicted_lev = []
    fp_sentences_lev = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_filtered:
            print("WRONG FILE")

        label_lev, score_lev, sentence_lev = fc.get_best_match_ner_lev(
            info["fact"], per_file_ner_filtered[actual_path], "og_ratio", thresh
        )
        predicted_lev.append(label_lev)
        if label_lev == 1 and gold_labels[i] == 0:
            fp_sentences_lev.append((info["fact"], sentence_lev, score_lev))

    if thresh == inspect_thresh_lev:
        fp_inspect_lev = fp_sentences_lev

    fc.report_at_threshold(gold_labels, predicted_lev, thresh)

# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_lev:
#     print(f"score={sc}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 5. Fuzzy similarity — RapidFuzz

In [ ]:
rf_chosen_thresh = [0, 50, 55, 60, 65, 70, 75, 80, 85, 90, 100]
inspect_thresh_rf = 55
fp_inspect_rf = []

for thresh in rf_chosen_thresh:
    predicted_rf = []
    fp_sentences_rf = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_filtered:
            print("WRONG FILE")

        label_rf, score_rf, sentence_rf = fc.get_best_match_ner_rapfuz(
            info["fact"], per_file_ner_filtered[actual_path], "token_sort", thresh
        )
        predicted_rf.append(label_rf)
        if label_rf == 1 and gold_labels[i] == 0:
            fp_sentences_rf.append((info["fact"], sentence_rf, score_rf))

    if thresh == inspect_thresh_rf:
        fp_inspect_rf = fp_sentences_rf

    fc.report_at_threshold(gold_labels, predicted_rf, thresh)
# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_rf:
#     print(f"score={sc}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 6. Embedding Similarity -> all-MiniLM-L6-v2

In [ ]:
# model used for Embedding Similarity Approach
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# building the embedding cache for the sentences using build_embedding_cache and save it.
# Then just load the pickle below.
embedding_cache_emb = fc.build_embedding_cache(per_file_ner_filtered, model, show_progress_bar=True)
with open("new_embedding_cache_emb.pkl", "wb") as f:
    pickle.dump(embedding_cache_emb, f)

In [ ]:
with open("new_embedding_cache_emb.pkl", "rb") as emb:
    embedding_cache_emb = pickle.load(emb)

In [ ]:
chosen_thresh_emb = [0.0, 0.2, 0.5, 0.55, 0.6, 0.62, 0.63, 0.64, 0.65, 0.67, 0.7, 0.75, 0.8, 0.85, 0.9, 1]
inspect_thresh_emb = 0.65
fp_inspect_emb = []

for thresh in chosen_thresh_emb:
    emb_predicted = []
    fp_sentences_emb = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_filtered:
            print("WRONG FILE")

        score_emb, matched_sentence_emb, found_emb = fc.get_first_above_thresh(
            info["fact"], per_file_ner_filtered[actual_path],
            embedding_cache_emb[actual_path], model, thresh
        )
        emb_predicted.append(1 if found_emb else 0)
        if found_emb and gold_labels[i] == 0:
            fp_sentences_emb.append((info["fact"], matched_sentence_emb, score_emb))

    if thresh == inspect_thresh_emb:
        fp_inspect_emb = fp_sentences_emb

    fc.report_at_threshold(gold_labels, emb_predicted, thresh)
    
# To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_emb:
#    print(f"score={sc:.4f}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 7. Paraphrase Detection —> paraphrase-mpnet-base-v2

In [ ]:
# model used for Paraphrase Detection Approach
model_paraph = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")

In [ ]:
# building the embedding cache for the sentences using build_embedding_cache and save it.
# Then just load the pickle below.
embedding_cache_paraph = fc.build_embedding_cache(per_file_ner_filtered, model_paraph, show_progress_bar=True)
with open("new_embedding_cache_paraph.pkl", "wb") as f:
     pickle.dump(embedding_cache_paraph, f)

In [ ]:
with open("new_embedding_cache_paraph.pkl", "rb") as paraph:
    embedding_cache_paraph = pickle.load(paraph)

In [ ]:
chosen_thresh_paraph = [0.0, 0.2, 0.5, 0.55, 0.6, 0.62, 0.63, 0.64, 0.65, 0.67, 0.7, 0.75, 0.77, 0.8, 0.82, 0.85, 0.9, 1]
inspect_thresh_paraph = 0.64
fp_inspect_paraph = []

for thresh in chosen_thresh_paraph:
    paraph_predicted = []
    fp_sentences_paraph = []
    for i, info in enumerate(info_results_json):
        actual_path = os.path.join(file_path, info["text_name"])
        if actual_path not in per_file_ner_filtered:
            print("WRONG FILE")

        score_paraph, matched_sentence_paraph, found_paraph = fc.get_first_above_thresh_paraph(
            info["fact"], per_file_ner_filtered[actual_path],
            embedding_cache_paraph[actual_path], model_paraph, thresh
        )
        paraph_predicted.append(1 if found_paraph else 0)
        if found_paraph and gold_labels[i] == 0:
            fp_sentences_paraph.append((info["fact"], matched_sentence_paraph, score_paraph))

    if thresh == inspect_thresh_paraph:
        fp_inspect_paraph = fp_sentences_paraph

    fc.report_at_threshold(gold_labels, paraph_predicted, thresh)
#To inspect false positives at the chosen threshold:
# for fact, sent, sc in fp_inspect_paraph:
#     print(f"score={sc:.4f}\n  fact: {fact}\n  sent: {sent}\n" + "-" * 80)

## 8. NLI —> Small Model cross-encoder/nli-deberta-v3-base

In [ ]:
# models used for NLI Approach
model_cross        = CrossEncoder("cross-encoder/nli-deberta-v3-base")
model_cross_advanc = CrossEncoder("MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli")

In [ ]:
# The two models order their labels differently. So the fc file uses explicit
# label-index maps (fc.NLI_DEBERTA / fc.MORITZLAURER) to capture them correctly.
print("nli-deberta :", model_cross.config.id2label)
print("MoritzLaurer:", model_cross_advanc.config.id2label)

In [ ]:
# Scoring every (sentence, fact) pair and checkpoint it.
# tqdm was used to show the progress and the score were saved every 50 facts to prevent loosing
# them in case of computer crash
checkpoint_file = "nli_small_sentence_no_softmax.pkl"
all_doc_names = [info["text_name"] for info in info_results_json]

try:
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    all_fact_data = checkpoint["all_fact_data"]
    gold_labels_ck = checkpoint["gold_labels"]
    start_idx = checkpoint["next_idx"]
    print(f"Resuming from fact {start_idx}")
except FileNotFoundError:
    all_fact_data, gold_labels_ck, start_idx = [], [], 0
    print("Starting fresh")

from tqdm import tqdm
for i in tqdm(range(start_idx, len(all_facts)), desc="Facts"):
    actual_path = os.path.join(file_path, all_doc_names[i])
    if actual_path not in per_file_ner_filtered:
        print(f"PATH IS NOT THERE for fact {i}")
        continue
    all_scores, all_sentences = fc.get_all_pairs_cross_encoder(
        all_facts[i], per_file_ner_filtered[actual_path], model_cross
    )
    all_fact_data.append({"fact_idx": i, "fact": all_facts[i],
                          "document": all_doc_names[i],
                          "sentences": all_sentences, "scores": all_scores})
    gold_labels_ck.append(gold_labels[i])
    if len(all_fact_data) % 50 == 0:
        with open(checkpoint_file, "wb") as f:
            pickle.dump({"all_fact_data": all_fact_data, "gold_labels": gold_labels_ck,
                         "next_idx": i + 1}, f)
        print(f"\nCheckpoint saved at fact {i}")

with open(checkpoint_file, "wb") as f:
    pickle.dump({"all_fact_data": all_fact_data, "gold_labels": gold_labels_ck,
                 "next_idx": len(all_facts)}, f)
print(f"\nDone. {len(all_fact_data)} facts processed.")

In [ ]:
with open("nli_small_sentence_no_softmax.pkl", "rb") as f:
    checkpoint2 = pickle.load(f)
    all_fact_data_2 = checkpoint2["all_fact_data"]
    gold_labels_all_cross_2 = checkpoint2["gold_labels"]

print("First pair scores:", all_fact_data_2[0]["scores"][0])
print("Sum:", sum(all_fact_data_2[0]["scores"][0]))   # to check the probabilities and see whether they are logits or not

# Early_exit (stop at first entailment) scoring (entailment is column 1 for nli-deberta)
thresholds = [0.0, 0.2, 0.25, 0.3, 0.33, 0.35, 0.4, 0.6, 0.8, 0.9, 1]
for thresh in thresholds:
    preds2 = [fc.predict_early_exit(entry, thresh, fc.NLI_DEBERTA["entailment"])
              for entry in all_fact_data_2]
    fc.report_at_threshold(gold_labels_all_cross_2, preds2, thresh, digits=3)

### False Positive Inspection (nli-deberta, t = 0.3)

In [ ]:
# export the nli-deberta false positives to CSV file for error-analysis
import csv

thresh = 0.3
with open("nli_false_positives.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["fact_idx", "document", "fact", "exit_sentence", "entail_prob"])
    for entry in all_fact_data_2:
        idx, prob = fc.early_exit(entry, thresh, fc.NLI_DEBERTA["entailment"])
        if idx is not None and gold_labels_all_cross_2[entry["fact_idx"]] == 0:
            writer.writerow([entry["fact_idx"], entry["document"], entry["fact"],
                             entry["sentences"][idx], round(prob, 4)])
print("Saved nli_false_positives.csv")

### Single Example Diagnostics and Sanity Check

In [ ]:
i = 65                                         # fact_idx 
entry = next(e for e in all_fact_data_2 if e["fact_idx"] == i)
j = 81                                         # exit_index from the CSV

sent = entry["sentences"][j]
fact = entry["fact"]
stored_logits = np.asarray(entry["scores"][j], dtype=float)
fresh_logits  = np.asarray(model_cross.predict([[sent, fact]])[0], dtype=float)  # same [sent, fact] order

print("stored logits:", np.round(stored_logits, 4))
print("fresh  logits:", np.round(fresh_logits, 4))
print("stored softmax:", np.round(softmax(stored_logits), 4))
print("fresh  softmax:", np.round(softmax(fresh_logits), 4))
print("sentence:", repr(sent))
print("fact    :", repr(fact))

In [ ]:
# export nli-deberta false positives with the (idx, score) helper at a chosen threshold
# this provides the exact match the model sees without needing to look at CSV
THRESHOLD = 0.3
rows = []
for entry, gold in zip(all_fact_data_2, gold_labels_all_cross_2):
    idx, score = fc.early_exit(entry, THRESHOLD, fc.NLI_DEBERTA["entailment"])
    if idx is not None and gold == 0:          # false positive
        rows.append({"fact": entry["fact"], "sentence": entry["sentences"][idx],
                     "score": round(score, 6)})

fp_df = pd.DataFrame(rows)
fp_df.to_csv(f"nli_false_positives_thr{THRESHOLD}.csv", index=False)
print(f"{len(fp_df)} false positives at threshold {THRESHOLD}")

In [ ]:
fact_query = "Here comes the fact."
entry = next(e for e in all_fact_data_2 if e["fact"] == fact_query)

# first sentence that crosses 0.3 — the actual early-exit premise
idx, prob = fc.early_exit(entry, 0.3, fc.NLI_DEBERTA["entailment"])

sent   = entry["sentences"][idx]
stored = np.asarray(entry["scores"][idx], dtype=float)
fresh  = np.asarray(model_cross.predict([[sent, fact_query]])[0], dtype=float)

print("exit_index :", idx)
print("sentence repr:", repr(sent))
print("sentence len :", len(sent))
print("stored softmax:", np.round(softmax(stored), 4))
print("fresh  softmax:", np.round(softmax(fresh), 4))

## 9. NLI —> Advanced Model MoritzLaurer DeBERTa-v3-large

Same decision functions; only the label-index map changes (`fc.MORITZLAURER`).

In [ ]:
with open("new-nli-everything-based-advance-no-softmax.pkl", "rb") as f3:
    checkpoint3 = pickle.load(f3)
all_fact_data_3 = checkpoint3["all_fact_data"]
gold_labels_all_cross_3 = checkpoint3["gold_labels"]

print("First pair scores:", all_fact_data_3[0]["scores"][0])
print("Sum:", sum(all_fact_data_3[0]["scores"][0]))   #  to check the probabilities and see whether they are logits or not

# early_exit scoring (entailment is column 0 for MoritzLaurer)
thresholds = [0.0, 0.1, 0.2, 0.25, 0.3, 0.33, 0.35, 0.37, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99, 1, 1.1]
for thresh_l in thresholds:
    preds_l = [fc.predict_early_exit(entry_l, thresh_l, fc.MORITZLAURER["entailment"])
              for entry_l in all_fact_data_3]
    fc.report_at_threshold(gold_labels_all_cross_3, preds_l, thresh_l)

### False Positive Inspection (MoritLaurer, t = 0.2)

In [ ]:
# export the MoritzLaurer false positives to CSV file for error-analysis
thresh_l = 0.2
with open("nli_large_false_positives.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["fact_idx", "document", "fact", "exit_sentence", "entail_prob"])
    for entry_l in all_fact_data_3:
        idx_l, prob_l = fc.early_exit(entry_l, thresh_l, fc.MORITZLAURER["entailment"])
        if idx_l is not None and gold_labels_all_cross_3[entry_l["fact_idx"]] == 0:
            writer.writerow([entry_l["fact_idx"], entry_l["document"], entry_l["fact"],
                             entry_l["sentences"][idx_l], round(prob_l, 4)])
print("Saved nli_large_false_positives.csv")

In [ ]:
# export advanced-model false positives at a chosen threshold
THRESHOLD = 0.2
rows = []
for entry_l, gold_l in zip(all_fact_data_3, gold_labels_all_cross_3):
    idx_l, score_l = fc.early_exit(entry_l, THRESHOLD, fc.MORITZLAURER["entailment"])
    if idx_l is not None and gold_l == 0:          # false positive
        rows.append({"fact": entry_l["fact"], "sentence": entry_l["sentences"][idx_l],
                     "score": round(score_l, 6)})

fp_df = pd.DataFrame(rows)
fp_df.to_csv(f"nli_false_positives_adv_thr{THRESHOLD}.csv", index=False)
print(f"{len(fp_df)} false positives at threshold {THRESHOLD}")

In [ ]:
fact_query = "Here comes the fact."
entry = next(entry for entry in all_fact_data_3 if entry["fact"] == fact_query)

idx, prob = fc.early_exit(entry, 0.2, fc.MORITZLAURER["entailment"])

sent   = entry["sentences"][idx]
stored = np.asarray(entry["scores"][idx], dtype=float)
fresh  = np.asarray(model_cross_advanc.predict([[sent, fact_query]])[0], dtype=float)

print("exit_index j:", idx)
print("sentence repr:", repr(sent))
print("sentence len :", len(sent))
print("stored softmax:", np.round(softmax(stored), 4))
print("fresh  softmax:", np.round(softmax(fresh), 4))

### Asymmetric Scoring (large model)

In [ ]:
thresholds = [0.0, 0.1, 0.2, 0.25, 0.3, 0.33, 0.35, 0.37, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.97, 0.99, 1, 1.1]
for thresh_l in thresholds:
    preds_l = [fc.predict_asymmetric(entry_l, thresh_l,
                                    fc.MORITZLAURER["entailment"],
                                    fc.MORITZLAURER["contradiction"])
              for entry_l in all_fact_data_3]
    fc.report_at_threshold(gold_labels_all_cross_3, preds_l, thresh_l)